# PH Anomaly Detection — BetaVAE Demo

End-to-end demonstration of the pulse-height anomaly detector on a PH feature cache.

**Pipeline:** L1 `dp_ph256` Zarr → `pa-prep-ph` (feature cache) → BetaVAE → reconstruction-error anomaly score

The VAE is **unsupervised**: it learns to reconstruct normal PH frames, so frames it
reconstructs poorly (high per-sample MSE) are the anomaly candidates.

This notebook keeps the *volatile* research code (training wrappers, plots) in `lab.py`
next to it, and the *trusted, tested* code in the `panoseti_analysis` package (the same
`fit` engine + VAE hooks that `pa-train-vae` runs on Ray). Edit `lab.py` freely — with
`%autoreload 2` below, changes hot-reload.

---

### Prerequisites

A PH feature cache built from L1 `dp_ph256` stores:

```bash
pa-prep-ph --stores /mnt/beegfs/runs/*/L1/*.dp_ph256.*.zarr \
    --out ml/anomaly-detection/data/ --recipe ml/anomaly-detection/recipes/vae_train_v1.yml
```

Then set `FEATURE_CACHE` below to the resulting `features.*.zarr` path.

In [ ]:
# panoseti_analysis is installed editable (uv sync); lab.py sits next to this notebook.
%load_ext autoreload
%autoreload 2

from pathlib import Path

import lab

from panoseti_analysis.paths import ML

device = lab.get_device()
print(f"device: {device}")

## §0 Configuration

In [ ]:
# Default: a feature cache under ml/anomaly-detection/data/. Override with any PH cache:
#   FEATURE_CACHE = Path("/mnt/beegfs/features/features.<hash>.zarr")
FEATURE_CACHE = ML / "anomaly-detection" / "data" / "features.zarr"

# Quick-iteration hyperparameters (override the package defaults). For a full run use the
# recipe (recipes/vae_train_v1.yml) via `pa-train-vae`.
HP = {"latent_dim": 32, "hidden_dim": 64, "beta": 4e-9, "epochs": 30, "batch_size": 256}

## §1 Train

`lab.quick_train_vae` loads the cache (carving out a seeded validation split), then runs the
shared `fit` engine with the BetaVAE hooks. It returns the *best-checkpoint* model and the
full `TrainResult` history.

In [ ]:
%%time
model, result = lab.quick_train_vae(FEATURE_CACHE, HP, device=device)
print(f"epochs run     : {len(result.history)}")
print(f"best val_loss  : {result.best_value:.4f}")

## §2 Training Curves

In [ ]:
lab.plot_history(result, keys=("train_loss", "val_loss", "val_recon_mse"))

## §3 Anomaly Scores (reconstruction error)

High per-sample reconstruction MSE → anomaly candidate. The tail of this histogram is
where to look for interesting PH frames.

In [ ]:
_x_train, x_val = lab.load_features(FEATURE_CACHE)
lab.plot_error_hist(model, x_val, device=device)

## §4 Reconstructions

In [ ]:
lab.plot_reconstructions(model, x_val, n=8, device=device)

## §5 Top Anomalies

The highest-error validation frames — the model's anomaly picks.

In [ ]:
import numpy as np

errors = lab.reconstruction_errors(model, x_val, device=device)
top = np.argsort(-errors)[:8]
print("top-8 anomaly indices:", top.tolist())
print("their reconstruction MSE:", np.round(errors[top], 4).tolist())
lab.plot_reconstructions(model, x_val[top], n=len(top), device=device)